# 2. Multi-Armed Bandits

Bu notebook, Sutton & Barto kitabının 2. bölümünü kapsar.

## İçindekiler
1. K-Armed Bandit Problemi
2. Action-Value Methods
3. Epsilon-Greedy Strategy
4. Upper Confidence Bound (UCB)
5. Gradient Bandit Algorithms

## 2.1 K-Armed Bandit Problemi

**Senaryo**: K tane slot makinesi var. Her birinin farklı (bilinmeyen) ödül dağılımı var.

**Amaç**: Toplam ödülü maksimize et.

**Dilemma**: 
- **Exploration**: Yeni aksiyonları dene, bilgi topla
- **Exploitation**: Şu ana kadar en iyi bilinen aksiyonu kullan

### Matematiksel Formülasyon

Her aksiyon $a$'nın **gerçek değeri**:
$$q_*(a) = E[R_t | A_t = a]$$

Bunu bilmiyoruz, o yüzden **tahmin** ediyoruz:
$$Q_t(a) \approx q_*(a)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class KArmedBandit:
    """K-Armed Bandit environment."""
    
    def __init__(self, k=10):
        self.k = k
        self.reset()
    
    def reset(self):
        # Her kolun gerçek değeri N(0,1)'den çekilir
        self.q_true = np.random.randn(self.k)
        self.optimal_action = np.argmax(self.q_true)
        return self.q_true.copy()
    
    def step(self, action):
        # Reward: gerçek değer + noise
        reward = self.q_true[action] + np.random.randn()
        is_optimal = (action == self.optimal_action)
        return reward, is_optimal

# Test
bandit = KArmedBandit(k=10)
print(f"Gerçek değerler: {bandit.q_true.round(2)}")
print(f"Optimal aksiyon: {bandit.optimal_action}")

In [ ]:
# Gerçek değerleri görselleştir
fig, ax = plt.subplots(figsize=(10, 5))

actions = range(bandit.k)
colors = ['green' if i == bandit.optimal_action else 'steelblue' for i in actions]

ax.bar(actions, bandit.q_true, color=colors, alpha=0.7)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Action')
ax.set_ylabel('True Value q*(a)')
ax.set_title('10-Armed Bandit: True Action Values')
ax.set_xticks(actions)
plt.show()

## 2.2 Action-Value Methods

### Sample-Average Method

Aksiyon değerini, o aksiyondan alınan reward'ların ortalaması olarak tahmin et:

$$Q_t(a) = \frac{\sum_{i=1}^{t-1} R_i \cdot \mathbb{1}_{A_i=a}}{\sum_{i=1}^{t-1} \mathbb{1}_{A_i=a}}$$

### Incremental Update

Her seferinde tüm geçmişi tutmak yerine, artımlı güncelleme:

$$Q_{n+1} = Q_n + \frac{1}{n}[R_n - Q_n]$$

Genel form:
$$NewEstimate = OldEstimate + StepSize \times [Target - OldEstimate]$$

In [ ]:
class BanditAgent:
    """Base class for bandit agents."""
    
    def __init__(self, k, initial_value=0.0):
        self.k = k
        self.initial_value = initial_value
        self.reset()
    
    def reset(self):
        self.Q = np.ones(self.k) * self.initial_value  # Value estimates
        self.N = np.zeros(self.k)  # Action counts
    
    def select_action(self):
        raise NotImplementedError
    
    def update(self, action, reward):
        self.N[action] += 1
        # Incremental mean update
        self.Q[action] += (reward - self.Q[action]) / self.N[action]

## 2.3 Epsilon-Greedy Strategy

**Greedy**: Her zaman en yüksek tahmin edilen değere sahip aksiyonu seç.
$$A_t = \arg\max_a Q_t(a)$$

**Epsilon-Greedy**: Çoğu zaman greedy, bazen rastgele explore et.

$$A_t = \begin{cases} \arg\max_a Q_t(a) & \text{with probability } 1-\epsilon \\ \text{random action} & \text{with probability } \epsilon \end{cases}$$

In [ ]:
class EpsilonGreedyAgent(BanditAgent):
    """Epsilon-greedy action selection."""
    
    def __init__(self, k, epsilon=0.1, initial_value=0.0):
        super().__init__(k, initial_value)
        self.epsilon = epsilon
    
    def select_action(self):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.k)  # Explore
        else:
            # Exploit (tie-breaking randomly)
            max_q = np.max(self.Q)
            max_actions = np.where(self.Q == max_q)[0]
            return np.random.choice(max_actions)

In [ ]:
def run_experiment(agent_class, bandit, n_steps=1000, n_runs=200, **agent_kwargs):
    """Birden fazla run üzerinden agent performansını ölç."""
    
    all_rewards = np.zeros((n_runs, n_steps))
    all_optimal = np.zeros((n_runs, n_steps))
    
    for run in range(n_runs):
        bandit.reset()
        agent = agent_class(bandit.k, **agent_kwargs)
        
        for step in range(n_steps):
            action = agent.select_action()
            reward, is_optimal = bandit.step(action)
            agent.update(action, reward)
            
            all_rewards[run, step] = reward
            all_optimal[run, step] = is_optimal
    
    return all_rewards.mean(axis=0), all_optimal.mean(axis=0)

In [ ]:
# Farklı epsilon değerlerini karşılaştır
bandit = KArmedBandit(k=10)
n_steps = 1000
n_runs = 200

epsilons = [0.0, 0.01, 0.1]
results = {}

for eps in epsilons:
    rewards, optimal = run_experiment(
        EpsilonGreedyAgent, bandit, n_steps, n_runs, epsilon=eps
    )
    results[eps] = {'rewards': rewards, 'optimal': optimal}
    print(f"ε={eps}: Avg reward = {rewards.mean():.3f}, Optimal % = {optimal.mean()*100:.1f}%")

In [ ]:
# Sonuçları görselleştir
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['red', 'green', 'blue']

for i, eps in enumerate(epsilons):
    axes[0].plot(results[eps]['rewards'], color=colors[i], label=f'ε={eps}')
    axes[1].plot(results[eps]['optimal'] * 100, color=colors[i], label=f'ε={eps}')

axes[0].set_xlabel('Steps')
axes[0].set_ylabel('Average Reward')
axes[0].set_title('Average Reward over Time')
axes[0].legend()

axes[1].set_xlabel('Steps')
axes[1].set_ylabel('% Optimal Action')
axes[1].set_title('Optimal Action Selection Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2.4 Upper Confidence Bound (UCB)

Epsilon-greedy rastgele explore eder. Daha akıllı bir yöntem: **belirsizliği** dikkate al.

$$A_t = \arg\max_a \left[ Q_t(a) + c \sqrt{\frac{\ln t}{N_t(a)}} \right]$$

- $Q_t(a)$: Exploitation terimi (mevcut tahmin)
- $c\sqrt{\frac{\ln t}{N_t(a)}}$: Exploration terimi (belirsizlik)
- Az denenen aksiyonlar daha yüksek bonus alır

In [ ]:
class UCBAgent(BanditAgent):
    """Upper Confidence Bound action selection."""
    
    def __init__(self, k, c=2.0, initial_value=0.0):
        super().__init__(k, initial_value)
        self.c = c
        self.t = 0
    
    def reset(self):
        super().reset()
        self.t = 0
    
    def select_action(self):
        self.t += 1
        
        # Henüz denenmemiş aksiyonları öncelikle dene
        if 0 in self.N:
            return np.where(self.N == 0)[0][0]
        
        # UCB formula
        ucb_values = self.Q + self.c * np.sqrt(np.log(self.t) / self.N)
        return np.argmax(ucb_values)

In [ ]:
# UCB vs Epsilon-Greedy karşılaştırması
bandit = KArmedBandit(k=10)

eps_rewards, eps_optimal = run_experiment(
    EpsilonGreedyAgent, bandit, n_steps, n_runs, epsilon=0.1
)

ucb_rewards, ucb_optimal = run_experiment(
    UCBAgent, bandit, n_steps, n_runs, c=2.0
)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(eps_rewards, label='ε-greedy (ε=0.1)', color='blue')
axes[0].plot(ucb_rewards, label='UCB (c=2)', color='orange')
axes[0].set_xlabel('Steps')
axes[0].set_ylabel('Average Reward')
axes[0].legend()
axes[0].set_title('UCB vs ε-greedy: Average Reward')

axes[1].plot(eps_optimal * 100, label='ε-greedy (ε=0.1)', color='blue')
axes[1].plot(ucb_optimal * 100, label='UCB (c=2)', color='orange')
axes[1].set_xlabel('Steps')
axes[1].set_ylabel('% Optimal Action')
axes[1].legend()
axes[1].set_title('UCB vs ε-greedy: Optimal Action %')

plt.tight_layout()
plt.show()

## 2.5 Gradient Bandit Algorithms

Değer tahmini yerine, **preference** (tercih) öğren:

$$\pi_t(a) = \frac{e^{H_t(a)}}{\sum_{b=1}^{k} e^{H_t(b)}}$$ (softmax)

Güncelleme (stochastic gradient ascent):

$$H_{t+1}(A_t) = H_t(A_t) + \alpha (R_t - \bar{R}_t)(1 - \pi_t(A_t))$$
$$H_{t+1}(a) = H_t(a) - \alpha (R_t - \bar{R}_t)\pi_t(a) \quad \forall a \neq A_t$$

In [ ]:
class GradientBanditAgent:
    """Gradient Bandit with softmax action selection."""
    
    def __init__(self, k, alpha=0.1, baseline=True):
        self.k = k
        self.alpha = alpha
        self.baseline = baseline
        self.reset()
    
    def reset(self):
        self.H = np.zeros(self.k)  # Preferences
        self.avg_reward = 0  # Baseline
        self.t = 0
    
    def softmax(self):
        exp_h = np.exp(self.H - np.max(self.H))  # Numerical stability
        return exp_h / np.sum(exp_h)
    
    def select_action(self):
        probs = self.softmax()
        return np.random.choice(self.k, p=probs)
    
    def update(self, action, reward):
        self.t += 1
        probs = self.softmax()
        
        baseline = self.avg_reward if self.baseline else 0
        
        # Update preferences
        one_hot = np.zeros(self.k)
        one_hot[action] = 1
        
        self.H += self.alpha * (reward - baseline) * (one_hot - probs)
        
        # Update baseline
        self.avg_reward += (reward - self.avg_reward) / self.t

In [ ]:
# Gradient Bandit with/without baseline
bandit = KArmedBandit(k=10)

# Run experiments
def run_gradient_experiment(bandit, n_steps=1000, n_runs=200, **kwargs):
    all_optimal = np.zeros((n_runs, n_steps))
    
    for run in range(n_runs):
        bandit.reset()
        # Shift rewards to make baseline matter
        bandit.q_true += 4
        
        agent = GradientBanditAgent(bandit.k, **kwargs)
        
        for step in range(n_steps):
            action = agent.select_action()
            reward, is_optimal = bandit.step(action)
            agent.update(action, reward)
            all_optimal[run, step] = is_optimal
    
    return all_optimal.mean(axis=0)

baseline_optimal = run_gradient_experiment(bandit, baseline=True, alpha=0.1)
no_baseline_optimal = run_gradient_experiment(bandit, baseline=False, alpha=0.1)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(baseline_optimal * 100, label='With baseline', color='blue')
plt.plot(no_baseline_optimal * 100, label='Without baseline', color='red')
plt.xlabel('Steps')
plt.ylabel('% Optimal Action')
plt.title('Gradient Bandit: Effect of Baseline')
plt.legend()
plt.show()

## Özet

Bu notebook'ta öğrendiklerimiz:

| Yöntem | Avantaj | Dezavantaj |
|--------|---------|------------|
| **Greedy** | Basit | Explore etmez |
| **ε-Greedy** | Basit, explore eder | Rastgele exploration |
| **UCB** | Akıllı exploration | Daha karmaşık |
| **Gradient Bandit** | Soft preferences | Baseline önemli |

### Sonraki Notebook
**03 - Markov Decision Processes**: Full RL problemi, states, transitions